# Notebook-1
Goal of this notebook is to Understand the data + reduce it to a form suitable for modeling without crashing memory.

#### Objectives
- Load datasets safely (especially studentVle.csv)
- Understand
    - What each file represents
    - Key features available
- Create student-level engagement signals
- Produce clean, aggregated datasets for later notebooks

# Imports & Global Setting

In [1]:
import pandas as pd
import gc

# Understand each file

| File                      | Meaning                                      |
| ------------------------- | -------------------------------------------- |
| `courses.csv`             | Course metadata                              |
| `studentInfo.csv`         | Student demographics + final result          |
| `studentRegistration.csv` | Enrollment & withdrawal dates                |
| `assessments.csv`         | Assessment definitions                       |
| `studentAssessment.csv`   | Student scores                               |
| `vle.csv`                 | Learning activities (videos, pages, forums)  |
| `studentVle.csv`          | **Student interactions with VLE (10M rows)** |


Your entire project revolves around : How students interact with learning material over time.

# Load small tables normally

In [2]:
from pathlib import Path


In [3]:
try:
    root_folder_path=Path(__file__).resolve().parent.parent.parent

except:
    root_folder_path=Path().resolve().parent.parent.parent

print(root_folder_path)

D:\AI-ML\REAL WORLD PROJECTS


In [4]:
# Change line 1 to this:
base_path = Path(root_folder_path) / "MACHINE LEARNING/Academic-Risk-Engagement-Prediction-System/data"
files_list=[p for p in Path(base_path).iterdir() if p.is_file()]
for file_path in files_list:
    print(file_path)

D:\AI-ML\REAL WORLD PROJECTS\MACHINE LEARNING\Academic-Risk-Engagement-Prediction-System\data\assessments.csv
D:\AI-ML\REAL WORLD PROJECTS\MACHINE LEARNING\Academic-Risk-Engagement-Prediction-System\data\courses.csv
D:\AI-ML\REAL WORLD PROJECTS\MACHINE LEARNING\Academic-Risk-Engagement-Prediction-System\data\studentAssessment.csv
D:\AI-ML\REAL WORLD PROJECTS\MACHINE LEARNING\Academic-Risk-Engagement-Prediction-System\data\studentInfo.csv
D:\AI-ML\REAL WORLD PROJECTS\MACHINE LEARNING\Academic-Risk-Engagement-Prediction-System\data\studentRegistration.csv
D:\AI-ML\REAL WORLD PROJECTS\MACHINE LEARNING\Academic-Risk-Engagement-Prediction-System\data\studentVle.csv
D:\AI-ML\REAL WORLD PROJECTS\MACHINE LEARNING\Academic-Risk-Engagement-Prediction-System\data\vle.csv


In [5]:
dataframes = {
    "courses" : pd.read_csv(base_path / "courses.csv"),
    "student_info" : pd.read_csv(base_path / "studentInfo.csv"),
    "student_reg" : pd.read_csv(base_path / "studentRegistration.csv"),
    "assessments" : pd.read_csv(base_path / "assessments.csv"),
}


In [6]:
for name,df in dataframes.items():
    print(f"--- Dataframe:'{name}' ----")
    print('#### DETAILS ####')
    display(df.info())
    print('#### 5 samples rows ####')
    display(df.head())

--- Dataframe:'courses' ----
#### DETAILS ####
<class 'pandas.DataFrame'>
RangeIndex: 22 entries, 0 to 21
Data columns (total 3 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   code_module                 22 non-null     str  
 1   code_presentation           22 non-null     str  
 2   module_presentation_length  22 non-null     int64
dtypes: int64(1), str(2)
memory usage: 836.0 bytes


None

#### 5 samples rows ####


,code_module,code_presentation,module_presentation_length
0,AAA,2013J,268
1,AAA,2014J,269
2,BBB,2013J,268
3,BBB,2014J,262
4,BBB,2013B,240


--- Dataframe:'student_info' ----
#### DETAILS ####
<class 'pandas.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   code_module           32593 non-null  str  
 1   code_presentation     32593 non-null  str  
 2   id_student            32593 non-null  int64
 3   gender                32593 non-null  str  
 4   region                32593 non-null  str  
 5   highest_education     32593 non-null  str  
 6   imd_band              31482 non-null  str  
 7   age_band              32593 non-null  str  
 8   num_of_prev_attempts  32593 non-null  int64
 9   studied_credits       32593 non-null  int64
 10  disability            32593 non-null  str  
 11  final_result          32593 non-null  str  
dtypes: int64(3), str(9)
memory usage: 4.8 MB


...

,code_module,code_presentation,id_student,date_registration,date_unregistration
0,AAA,2013J,11391,-159.0,NaN
1,AAA,2013J,28400,-53.0,NaN
2,AAA,2013J,30268,-92.0,12.0
3,AAA,2013J,31604,-52.0,NaN
4,AAA,2013J,32885,-176.0,NaN


--- Dataframe:'assessments' ----
#### DETAILS ####
<class 'pandas.DataFrame'>
RangeIndex: 206 entries, 0 to 205
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   code_module        206 non-null    str    
 1   code_presentation  206 non-null    str    
 2   id_assessment      206 non-null    int64  
 3   assessment_type    206 non-null    str    
 4   date               195 non-null    float64
 5   weight             206 non-null    float64
dtypes: float64(2), int64(1), str(3)
memory usage: 12.0 KB


None

#### 5 samples rows ####


,code_module,code_presentation,id_assessment,assessment_type,date,weight
0,AAA,2013J,1752,TMA,19.0,10.0
1,AAA,2013J,1753,TMA,54.0,20.0
2,AAA,2013J,1754,TMA,117.0,20.0
3,AAA,2013J,1755,TMA,166.0,20.0
4,AAA,2013J,1756,TMA,215.0,30.0


# Load the big one : studentVle.csv

In [7]:
student_vle_with_optimization=pd.read_csv(base_path/'studentVle.csv')

In [8]:
memory_used_mb=student_vle_with_optimization.memory_usage(deep=True).sum()/1024**2
print(f"Memory used in MD : {memory_used_mb}")

Memory used in MD : 569.0534706115723


In [9]:
student_vle_with_optimization.info()

<class 'pandas.DataFrame'>
RangeIndex: 10655280 entries, 0 to 10655279
Data columns (total 6 columns):
 #   Column             Dtype
---  ------             -----
 0   code_module        str  
 1   code_presentation  str  
 2   id_student         int64
 3   id_site            int64
 4   date               int64
 5   sum_click          int64
dtypes: int64(4), str(2)
memory usage: 569.1 MB


In [10]:
del student_vle_with_optimization

In [11]:
gc.collect()

0

In [12]:
student_vle_dtypes={
    'code_module':'category',
    'code_presentation':'category',
    'id_student':'int32',
    'id_site':'int32',
    'date':'int16',
    'sumclick':'int16'
}

In [13]:
student_vle=pd.read_csv(base_path/'studentVle.csv',dtype=student_vle_dtypes)

In [14]:
memory_used_mb=student_vle.memory_usage(deep=True).sum()/1024**2
print(f"Memory used in MD :{memory_used_mb:.2f}")

Memory used in MD :203.23


# Understanding studentVle Features

In [15]:
student_vle.head(10)

,code_module,code_presentation,id_student,id_site,date,sum_click
0,AAA,2013J,28400,546652,-10,4
1,AAA,2013J,28400,546652,-10,1
2,AAA,2013J,28400,546652,-10,1
3,AAA,2013J,28400,546614,-10,11
4,AAA,2013J,28400,546714,-10,1
5,AAA,2013J,28400,546652,-10,8
6,AAA,2013J,28400,546876,-10,2
7,AAA,2013J,28400,546688,-10,15
8,AAA,2013J,28400,546662,-10,17
9,AAA,2013J,28400,546890,-10,1


In [16]:
student_vle.tail(10)

,code_module,code_presentation,id_student,id_site,date,sum_click
10655270,GGG,2014J,642694,896943,269,2
10655271,GGG,2014J,646169,896943,269,1
10655272,GGG,2014J,643672,896943,269,2
10655273,GGG,2014J,644226,896943,269,1
10655274,GGG,2014J,679769,896943,269,2
10655275,GGG,2014J,675811,896943,269,3
10655276,GGG,2014J,675578,896943,269,1
10655277,GGG,2014J,654064,896943,269,3
10655278,GGG,2014J,654064,896939,269,1
10655279,GGG,2014J,654064,896939,269,1


# Early Aggregation

In [17]:
# groupby aggregation
# New dataframe, original rows are not modified

student_engagement = (
    student_vle
    .groupby("id_student")
    . agg(
        total_click=("sum_click", "sum"),
        avg_clicks_per_day=("sum_click", "mean"),
        max_clicks_day=("sum_click", "max"),
        active_days=("date", "nunique")
    )
    .reset_index()
)

In [18]:
del student_vle
gc.collect()

0

In [19]:
student_engagement.head()

,id_student,total_click,avg_clicks_per_day,max_clicks_day,active_days
0,6516,2791,4.216012,49,159
1,8462,656,2.157895,16,56
2,11391,934,4.765306,76,40
3,23629,161,2.728814,13,16
4,23698,910,2.983607,78,70


In [20]:
student_engagement.info()

<class 'pandas.DataFrame'>
RangeIndex: 26074 entries, 0 to 26073
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_student          26074 non-null  int32  
 1   total_click         26074 non-null  int64  
 2   avg_clicks_per_day  26074 non-null  float64
 3   max_clicks_day      26074 non-null  int64  
 4   active_days         26074 non-null  int64  
dtypes: float64(1), int32(1), int64(3)
memory usage: 916.8 KB


In [21]:
dataframes['student_info'].head(2)

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass


In [22]:
student_master=(
    dataframes['student_info'].merge(student_engagement,on='id_student',how='left')
)

In [23]:
student_master.head(2)

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,total_click,avg_clicks_per_day,max_clicks_day,active_days
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,934.0,4.765306,76.0,40.0
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass,1435.0,3.337209,23.0,80.0


# Basic Sanity Checks

In [24]:
student_master.isnull().mean().sort_values(ascending=False)*100

total_click             8.750345
avg_clicks_per_day      8.750345
max_clicks_day          8.750345
active_days             8.750345
imd_band                3.408707
code_module             0.000000
id_student              0.000000
code_presentation       0.000000
age_band                0.000000
highest_education       0.000000
region                  0.000000
gender                  0.000000
final_result            0.000000
disability              0.000000
studied_credits         0.000000
num_of_prev_attempts    0.000000
dtype: float64

- 8.75% of student never interacted with VLE
- 3.4% missing imd_band --> socio-sconomic data
- Everything else = 0% --> Very clean dataset overall

In [25]:
root_folder_path

WindowsPath('D:/AI-ML/REAL WORLD PROJECTS')

In [26]:
output_path = Path(root_folder_path) / "MACHINE LEARNING/Academic-Risk-Engagement-Prediction-System/Dataset" / "student_master_v1.csv"
student_master.to_csv(output_path, index=False)
